# Correctness evaluation — Layered numerical evidence

Run cells in order, or restart the kernel and **Run All**. Helpers live in
`experiments/homework1/evaluation`; generated evidence stays under ignored `outputs/`.
Every run gets a unique directory. No model downloads are performed.

Before launching Jupyter, activate your project Python environment, source CANN's `set_env.sh`, and set
`ASCEND_RT_VISIBLE_DEVICES` to **one free physical chip**. The native subprocess
uses that chip; the independent Transformers reference uses CPU. The notebook
does not attach to or stop existing servers.

Defaults cover lengths 1, 3, 4, 5, 127, 128, 129, 512, 2048 plus four chat prompts.
To execute a smaller *explicit* matrix, set `HOMEWORK_EVAL_LENGTHS=1,3,129` before
starting the kernel. The manifest always records the actual matrix. This is
correctness evaluation; timing these instrumented runs is not a performance result.

## 1. Configuration, provenance, and predeclared criteria

Keep these limits fixed while evaluating. CPU BF16 versus CPU FP32 provides an
independent rounding baseline before serving comparisons. These initial screening
limits account for BF16 inputs/weights and accumulated error, and are deliberately
separate from the tighter FP32 operator tests. They are not a universal accuracy
guarantee. Investigate baseline failures or adaptation errors; never automatically
increase a threshold. All four numerical criteria must pass.

In [ ]:
import dataclasses
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "python/tokenspeed").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if (
    not os.environ.get("ASCEND_RT_VISIBLE_DEVICES")
    or "," in os.environ["ASCEND_RT_VISIBLE_DEVICES"]
):
    raise RuntimeError(
        "Set ASCEND_RT_VISIBLE_DEVICES to one free chip before starting the kernel"
    )

import torch
from transformers import AutoTokenizer
from experiments.homework1.evaluation.fixtures import (
    build_fixtures,
    shared_eos_ids,
    environment_manifest,
    save_fixtures,
)
from experiments.homework1.evaluation.metrics import Limits, RunArtifacts
from experiments.homework1.evaluation.model import (
    Reference,
    compare_generation,
    compare_lifecycle,
    compare_teacher,
    native_run,
    rounding_baseline,
)
from experiments.homework1.evaluation.gdn import compare_layer
from experiments.homework1.evaluation.report import finish, operator_tests

CHECKPOINT = Path(
    os.environ.get("HOMEWORK_EVAL_MODEL", ROOT / "outputs/models/Qwen3.5-0.8B")
).resolve()
LENGTHS = [
    int(x)
    for x in os.environ.get(
        "HOMEWORK_EVAL_LENGTHS", "1,3,4,5,127,128,129,512,2048"
    ).split(",")
]
TEACHER_STEPS = int(os.environ.get("HOMEWORK_EVAL_TEACHER_STEPS", "4"))
FREE_STEPS = int(os.environ.get("HOMEWORK_EVAL_FREE_STEPS", "16"))
INCLUDE_CHAT = os.environ.get("HOMEWORK_EVAL_INCLUDE_CHAT", "1") == "1"
SEED = 17
THREADS = 8
CONTEXT_LENGTH = 8192
CAPACITY = 32768  # Multiple of the real 1024-token LCM parent, scheduler reserve plus two 2048-token requests.
TIMEOUT_SECONDS = 1800
REUSE_ATTEMPTS = 32
LIMITS = {
    "logits": Limits(atol=0.6, rtol=0.03, relative_l2=0.06, rmse=0.18),
    "layer": Limits(atol=0.08, rtol=0.05, relative_l2=0.06, rmse=0.03),
    "ssm": Limits(atol=0.02, rtol=0.05, relative_l2=0.08, rmse=0.02),
    "conv": Limits(atol=0.03, rtol=0.03, relative_l2=0.04, rmse=0.02),
    "lifecycle": Limits(atol=0.03, rtol=0.01, relative_l2=0.02, rmse=0.01),
}
torch.manual_seed(SEED)
torch.set_num_threads(THREADS)
settings = dict(
    lengths=LENGTHS,
    teacher_steps=TEACHER_STEPS,
    free_steps=FREE_STEPS,
    include_chat=INCLUDE_CHAT,
    seed=SEED,
    threads=THREADS,
    context_length=CONTEXT_LENGTH,
    capacity=CAPACITY,
    reuse_attempts=REUSE_ATTEMPTS,
    failure_policy="record",
    native_dtype="BF16",
    ssm_dtype="FP32",
    native_device="Ascend",
    reference_dtype="FP32",
    reference_device="CPU",
    reference_attention="eager",
    native_attention="mha",
    sampling="greedy",
    graphs=False,
    prefix_caching=False,
    snapshot_pool=False,
    thinking=False,
    teacher_ignore_eos=True,
    free_ignore_eos=False,
    padding="packed unpadded inputs; causal attention; positions captured per step",
    limits={key: dataclasses.asdict(value) for key, value in LIMITS.items()},
)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
run = RunArtifacts(
    ROOT / "outputs/homework1/correctness/runs",
    run_id=run_id,
    manifest=environment_manifest(ROOT, CHECKPOINT, seed=SEED, settings=settings),
)
print(run.path)
print(json.dumps(settings, indent=2))

## 2. Save exact fixtures after chat formatting

Controlled lengths repeat **token IDs**, not words. Chat inputs include Chinese,
English, arithmetic and a repeated pattern, with thinking explicitly disabled.
The saved continuation IDs are supplied to both implementations during teacher
forcing. Fixtures and positions are available independently of notebook outputs.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, local_files_only=True)
checkpoint_config = json.loads((CHECKPOINT / "config.json").read_text())
EOS_TOKEN_IDS = shared_eos_ids(
    tokenizer.eos_token_id, checkpoint_config["text_config"]["eos_token_id"]
)
run.write_json(
    "generation_policy.json",
    dict(
        eos_token_ids=EOS_TOKEN_IDS,
        teacher_ignore_eos=True,
        free_ignore_eos=False,
        max_free_steps=FREE_STEPS,
    ),
)
fixtures = build_fixtures(
    tokenizer,
    lengths=LENGTHS,
    continuation_steps=TEACHER_STEPS,
    include_chat=INCLUDE_CHAT,
)
save_fixtures(run.path / "fixtures.json", fixtures)
run.write_json(
    "tokenizer.json",
    dict(
        special_tokens=tokenizer.special_tokens_map,
        all_special_ids=tokenizer.all_special_ids,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        thinking=False,
        add_generation_prompt=True,
        add_special_tokens=False,
    ),
)
for fixture in fixtures:
    print(
        f"{fixture.name}: {len(fixture.input_ids)} tokens; teacher IDs={fixture.continuation_ids}"
    )
if len(fixtures) < 2:
    raise ValueError("Choose at least two distinct fixtures for request isolation")
if (
    max(len(f.input_ids) for f in fixtures) + max(TEACHER_STEPS, FREE_STEPS)
    > CONTEXT_LENGTH
):
    raise ValueError("Fixture plus continuation exceeds the explicit context length")

## 3. Individual operators and state

Reuse the existing behavior tests instead of copying formulas into notebook cells.
This includes recurrence, convolution, packing/gates, normalization, state indexing
and padding. Full test names, failures and skips go to JUnit and the saved log.
A skipped NPU suite does not count as passing evidence.

In [ ]:
operators = operator_tests(ROOT, run, timeout_seconds=TIMEOUT_SECONDS)
print(json.dumps(operators, indent=2))

## 4. Independent CPU reference rounding baseline

Load the same local checkpoint twice, for true FP32 and BF16 eager computation.
Use cached teacher forcing for both. This measures the cumulative effect of BF16
computation, rather than rounding final FP32 logits. Baseline results use the
predeclared logit limits and stay visible alongside adaptation comparisons.

In [ ]:
reference = Reference(CHECKPOINT, threads=THREADS, dtype=torch.float32)
rounded_reference = Reference(CHECKPOINT, threads=THREADS, dtype=torch.bfloat16)
rounding_baseline(
    run,
    reference,
    rounded_reference,
    fixtures,
    steps=TEACHER_STEPS,
    limits=LIMITS["logits"],
)
del rounded_reference
print(run.summary())

## 5. Real native inference and complete GDN decoder layer

One disposable Engine runs independent fixtures, a two-request batch and a fresh
repeat. Startup dummy batches are excluded. Evaluation hooks capture actual
first-layer inputs, complete decoder outputs, convolution history, FP32 recurrent
state and physical block IDs. Teacher forcing changes only sampled token IDs;
attention, recurrence, cache allocation and writes use production code.

Compare the real first decoder layer against the independent HF layer, feeding
identical captured layer inputs with an independent cache per fixture. HF stores
recurrent state as `[H,K,V]`; serving uses `[H,V,K]`. HF retains four convolution
samples, while serving retains the last three needed for the next token.

In [ ]:
native = native_run(
    ROOT,
    CHECKPOINT,
    run.path / "native",
    fixtures,
    teacher_steps=TEACHER_STEPS,
    free_steps=FREE_STEPS,
    capacity=CAPACITY,
    reuse_attempts=REUSE_ATTEMPTS,
    failure_policy="record",
    seed=SEED,
    eos_token_ids=EOS_TOKEN_IDS,
    context_length=CONTEXT_LENGTH,
    timeout_seconds=TIMEOUT_SECONDS,
    lifecycle=True,
)
native_status = json.loads((native / "status.json").read_text())
if not native_status["completed"]:
    run.record_failure(
        "native_worker",
        "Native process failed; inspect native/worker.log",
        details=native_status,
    )
compare_layer(
    run,
    native,
    reference,
    fixtures,
    activation_limits=LIMITS["layer"],
    state_limits=LIMITS["ssm"],
    conv_limits=LIMITS["conv"],
)
print(run.summary())

## 6. Whole-model prefill and cached teacher-forced decode

Step zero compares prefill logits. Later steps must report native cached decode
and consume exactly the saved next-token IDs. The reference receives captured
native positions for numerical alignment. Separate zero-tolerance rows check
canonical text positions, so a runtime offset cannot be hidden. A growing-prefix prefill is rejected
as a decode substitute. Raw and vocabulary-mean-centered logits are both saved;
centering is a diagnostic for constant offsets, not a replacement for raw metrics.

In [ ]:
compare_teacher(
    run, native, reference, fixtures, steps=TEACHER_STEPS, limits=LIMITS["logits"]
)
print(run.summary())

## 7. Request isolation, state continuity and observed page reuse

Compare the actual batched A/B requests and fresh A rerun against their independent
runs, for every saved step and state. Capture real source/destination blocks. The
report explicitly distinguishes observed physical reuse from an unverified reuse
case. Prefix caching, retraction and distributed transfer are separate validations.

In [ ]:
lifecycle = compare_lifecycle(run, native, fixtures, limits=LIMITS["lifecycle"])
print(json.dumps(lifecycle, indent=2))

## 8. Free greedy generation and final report

Free generation uses the same explicit union of tokenizer/model EOS IDs, independently chosen tokens and a fixed
maximum length. Save the first differing token and the reference's top-two logit
margin at that point. After a divergence, the prefixes differ; later numerical
scores must not be presented as teacher-forced comparisons. Exact generation
agreement is a separate observation, not a replacement for numerical accuracy.

Failures remain in CSV, JSON and tensor artifacts. The final status does not hide
failed rows, skipped operators or unobserved physical reuse. Read `report.md` for
the compact table and `generation.json` for decoded representative outputs.

In [ ]:
generation = compare_generation(
    run,
    native,
    reference,
    fixtures,
    steps=FREE_STEPS,
    tokenizer=tokenizer,
    eos_token_ids=EOS_TOKEN_IDS,
)
summary = finish(run, operators=operators, generation=generation, lifecycle=lifecycle)
print(json.dumps(summary, indent=2))
for row in generation:
    print(
        row["fixture"],
        "first difference:",
        row["first_difference"],
        "margin:",
        row["reference_margin_at_difference"],
    )
    print("Native:", row["native_text"])
    print("Reference:", row["reference_text"])
failed = [row["name"] for row in run.rows if not row["passed"]]
print("Failed numerical comparisons:", failed)
print("Unavailable/failed checks:", run.failures)
print(
    "Required checks:",
    (
        "PASS"
        if summary["all_required_passed"]
        else "FAIL / INCOMPLETE — inspect saved evidence"
    ),
)
print("Artifacts:", run.path)